# Jev (System One) + LLM: classificar primeiro, responder depois

Este notebook mostra um padrão simples de arquitetura com **modelos System One**, usando o **Jev da TypeSafe** como exemplo.

**A ideia:** em vez de pedir para um LLM generalista fazer tudo (entender, classificar, decidir o tom e responder), dividimos o trabalho:

```
 ticket (texto)
      │
      ▼
┌──────────────┐   saída tipada    ┌──────────────────┐
│     Jev      │ ────────────────► │  LLM (OpenAI)    │ ──► resposta ao cliente
│ classificador│  departamento     │  gera a resposta │
└──────────────┘  frustração       │  guiado pelos    │
                 urgência          │  sinais tipados  │
                                   └──────────────────┘
```

- **Jev** responde perguntas sobre o texto e devolve **valores tipados e calibrados** (um `choice`, um `score`, uma probabilidade `noul`), não texto livre.
- **OpenAI** só se preocupa em escrever bem, já sabendo o que o Jev concluiu.

Referência: [Quickstart do Jev](https://docs.typesafe.ai/introduction/quickstart)

## 1. Setup

Antes de rodar, adicione duas chaves em **🔑 Secrets** (menu lateral esquerdo do Colab) e ative o acesso do notebook a elas:

- `TYPESAFE_API_KEY` → gere em https://console.typesafe.ai/keys
- `OPENAI_API_KEY` → gere em https://platform.openai.com/api-keys

In [ ]:
!pip install -q typesafe-sdk openai

In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    os.environ["TYPESAFE_API_KEY"] = userdata.get("TYPESAFE_API_KEY")
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    # Fallback caso os Secrets não estejam configurados
    os.environ.setdefault("TYPESAFE_API_KEY", getpass("TYPESAFE_API_KEY: "))
    os.environ.setdefault("OPENAI_API_KEY", getpass("OPENAI_API_KEY: "))

print("Chaves carregadas.")

## 2. As perguntas para o Jev

O Jev trabalha com três tipos de pergunta (*primitives*):

| Tipo | Para quê | Retorna |
|---|---|---|
| `Choice` | Escolher uma opção de uma lista | `choice`, `probabilities`, `confidence` |
| `Score` | Pontuar o texto em uma rubrica | `score`, `probabilities`, `confidence` |
| `Noul` | "Esta afirmação é verdadeira?" | `noul` (0 a 1) |

Vamos usar um de cada, para um cenário de **suporte ao cliente**:

In [ ]:
from typesafe_sdk import Choice, Noul, Score, TypeSafeClient

QUESTIONS = {
    "department": Choice(
        instructions="Qual departamento deve tratar este caso",
        criteria={
            "billing": "Problemas de pagamento ou de assinatura",
            "technical": "Bugs ou problemas de integração",
            "sales": "Dúvidas sobre preços ou sobre a conta",
        },
    ),
    "frustration": Score(
        instructions="O quão frustrado o cliente parece estar",
        criteria=[
            "Calmo, apenas relatando fatos",
            "Frustrado, mas educado",
            "Muito irritado, com linguagem forte",
        ],
    ),
    "is_urgent": Noul(
        instructions="A mensagem transmite urgência ou sensibilidade a prazo",
    ),
}

## 3. O classificador: saída tipada

Encapsulamos a chamada ao Jev numa função que devolve um `dataclass`. Assim, o resto do sistema não precisa saber nada sobre a resposta crua da API: recebe um objeto com campos e tipos bem definidos.

In [ ]:
from dataclasses import dataclass, asdict

@dataclass(frozen=True)
class Triage:
    department: str               # "billing" | "technical" | "sales"
    department_confidence: float  # 0..1
    frustration: float            # 0..2  (0 = calmo, 2 = muito irritado)
    urgency: float                # 0..1  (probabilidade de ser urgente)


def classify(ticket: str, client: TypeSafeClient) -> Triage:
    response = client.system_one(state=ticket, questions=QUESTIONS)
    dept = response.choices["department"]
    return Triage(
        department=dept.choice,
        department_confidence=dept.confidence,
        frustration=response.scores["frustration"].score,
        urgency=response.nouls["is_urgent"].noul,
    )

In [ ]:
TICKETS = {
    "api_fora": (
        "Nossa integração via API começou a retornar erro 500 em todas as requisições "
        "há cerca de 20 minutos, e não conseguimos processar nenhum pedido dos nossos "
        "clientes até que isso seja resolvido."
    ),
    "cobranca": (
        "Olá, percebi que fui cobrado duas vezes pela minha assinatura neste mês. "
        "Vocês poderiam verificar e estornar a cobrança duplicada quando puderem? Obrigado!"
    ),
    "irritado": (
        "Esta é a TERCEIRA vez que escrevo sobre isso. A plataforma de vocês é uma piada "
        "e vou cancelar tudo se ninguém me responder hoje."
    ),
    "comercial": (
        "Olá! Somos um time de 40 pessoas e gostaríamos de saber quanto custa o plano "
        "enterprise e se existe desconto para pagamento anual."
    ),
}

client = TypeSafeClient(model="jev-latest")

for name, text in TICKETS.items():
    print(f"{name:10s} → {classify(text, client)}")

Repare que os resultados são **números e categorias**, não frases. Isso é o que permite escrever lógica determinística (`if`, limiares, roteamento) em volta do modelo.

## 4. Transformando sinais tipados em instruções para o LLM

Aqui está o ponto central: o código traduz a saída do Jev em **regras explícitas** no prompt. O LLM não precisa *adivinhar* que o cliente está irritado ou que o caso é urgente; ele recebe isso como fato.

In [ ]:
import json

DEPARTMENT_GUIDANCE = {
    "billing": "Você atua pelo time de Financeiro. Peça dados da fatura/transação se necessário e nunca prometa reembolso sem verificação.",
    "technical": "Você atua pelo time de Suporte Técnico. Peça logs, horário do erro e passos para reproduzir, se faltarem.",
    "sales": "Você atua pelo time Comercial. Seja consultivo e ofereça uma conversa com um especialista.",
}

def tone_instruction(frustration: float) -> str:
    if frustration >= 1.4:
        return "O cliente está muito irritado. Comece reconhecendo o problema e pedindo desculpas de forma sincera, sem ser defensivo. Seja direto e ofereça próximo passo concreto."
    if frustration >= 0.6:
        return "O cliente está frustrado, mas educado. Demonstre empatia brevemente e vá logo à solução."
    return "O cliente está calmo. Tom cordial e objetivo."

def build_system_prompt(triage: Triage) -> str:
    parts = [
        "Você é um agente de atendimento ao cliente. Responda no mesmo idioma do cliente, em no máximo 120 palavras.",
        DEPARTMENT_GUIDANCE[triage.department],
        tone_instruction(triage.frustration),
    ]
    if triage.urgency >= 0.7:
        parts.append("O caso é URGENTE: diga explicitamente que está sendo priorizado e informe que um humano da equipe vai acompanhar.")
    if triage.department_confidence < 0.6:
        parts.append("A classificação do departamento é incerta: faça UMA pergunta curta para esclarecer o assunto antes de prometer qualquer ação.")

    parts.append(f"Sinais da triagem (JSON, gerados por um classificador):\n{json.dumps(asdict(triage), indent=2)}")
    return "\n\n".join(parts)

# Exemplo: o que o LLM vai receber para o ticket irritado
print(build_system_prompt(classify(TICKETS["irritado"], client)))

## 5. Pipeline completo: Jev → OpenAI

In [ ]:
from openai import OpenAI

openai_client = OpenAI()
OPENAI_MODEL = "gpt-4o-mini"

def handle_ticket(ticket: str) -> dict:
    # Classifica o ticket usando um modelo Sistema 1
    triage = classify(ticket, client)

    # Após a triagem é possível definir uma série de ações derivadas dos
    # resultados. Neste exemplo simples vamos apenas transferir as informações
    # para uma LLM (Sistema 2) responder
    completion = openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": build_system_prompt(triage)},
            {"role": "user", "content": ticket},
        ],
    )
    return {"triage": triage, "reply": completion.choices[0].message.content}


for name, text in TICKETS.items():
    result = handle_ticket(text)
    print("=" * 80)
    print(f"[{name}]  {text}\n")
    print("TRIAGEM:", result["triage"], "\n")
    print("RESPOSTA:", result["reply"], "\n")

### 5.1 Teste você mesmo!

Descomente as linhas da célula abaixo para interagir com a pipeline. A cada execução da célula você pode digitar uma issue diferente.

In [ ]:

# user_msg = input("Digite seu ticket: ")
# print()
# response = handle_ticket(user_msg)
# print(f"TRIAGEM: {response['triage']}\n")
# print("RESPOSTA:", response["reply"])


## 6. Comparação: Jev vs. LLM com saída estruturada

Até aqui usamos o Jev para classificar. Mas uma alternativa muito comum hoje é pedir ao próprio LLM para classificar, usando **Structured Outputs** (o modelo é forçado a devolver um JSON que respeita um schema Pydantic).

Vamos comparar as duas abordagens **fazendo as mesmas três perguntas** (departamento, frustração, urgência) sobre os mesmos tickets, em três dimensões:

1. **Resultados:** acerto contra um gabarito e concordância entre as abordagens.
2. **Consumo de tokens:** entrada e saída por chamada.
3. **Tempo de processamento:** latência por chamada.

> ⚠️ **Sobre o gabarito:** os 16 tickets abaixo foram rotulados à mão para este exemplo, e alguns são propositalmente ambíguos (principalmente frustração 1 vs. 2, e billing vs. sales). É uma amostra pequena, boa para ilustrar o método e não para concluir qual abordagem é "melhor" em geral. Para uma conclusão real, use tickets e rótulos do seu domínio.

### 6.1 Dataset com gabarito

In [ ]:
import time
import numpy as np
import pandas as pd

# gold_frus: 0 = calmo, 1 = frustrado mas civilizado, 2 = muito irritado
LABELED_TICKETS = [
    ("Nossa integração via API começou a retornar erro 500 em todas as requisições há cerca de 20 minutos, e não conseguimos processar nenhum pedido dos nossos clientes até que isso seja resolvido.", "technical", 0, True),
    ("Olá, percebi que fui cobrado duas vezes pela minha assinatura neste mês. Vocês poderiam verificar e estornar a cobrança duplicada quando puderem? Obrigado!", "billing", 0, False),
    ("Esta é a TERCEIRA vez que escrevo sobre essa cobrança duplicada. O financeiro de vocês é uma piada e vou cancelar tudo se ninguém me reembolsar hoje.", "billing", 2, True),
    ("Olá! Somos um time de 40 pessoas e gostaríamos de saber quanto custa o plano enterprise e se existe desconto para pagamento anual.", "sales", 0, False),
    ("O botão de exportar para CSV não faz nada quando eu clico. Não é nada grave, mas seria bom se fosse corrigido em algum momento deste mês.", "technical", 0, False),
    ("Estou tentando fazer login há uma hora e sempre recebo 'token inválido'. Tenho uma demo com um cliente em 30 minutos e estou ficando bem irritado.", "technical", 1, True),
    ("Posso mudar do plano mensal para o anual e manter o meu preço atual?", "sales", 0, False),
    ("Meu cartão foi recusado, mas o dinheiro saiu da minha conta mesmo assim. O QUE ESTÁ ACONTECENDO?? Resolvam isso AGORA.", "billing", 2, True),
    ("Estamos avaliando o produto de vocês para a nossa empresa de 200 pessoas. Vocês oferecem SSO e descontos por volume?", "sales", 0, False),
    ("As novas tentativas dos webhooks estão chegando fora de ordem, o que está quebrando a nossa conciliação. Não é urgente, mas gostaríamos de uma solução alternativa.", "technical", 0, False),
    ("A última atualização de vocês quebrou o nosso dashboard e, sinceramente, estou de saco cheio. É a terceira queda neste trimestre. Precisamos disso resolvido hoje ou vamos avaliar a concorrência.", "technical", 2, True),
    ("Por favor, me enviem a nota fiscal de março; o nosso time de contabilidade precisa dela até o fim do mês.", "billing", 0, False),
    ("Me disseram que o preço cairia após o período de teste, mas estou vendo o valor integral no meu cartão. Não foi isso que o vendedor prometeu.", "billing", 1, False),
    ("Obrigado pela correção rápida de ontem, está tudo funcionando agora! Só queria saber se existe desconto para organizações sem fins lucrativos.", "sales", 0, False),
    ("A sincronização do banco de dados de produção está fora do ar há 2 horas e os clientes estão reclamando. Por favor, escalem isso imediatamente!", "technical", 1, True),
    ("Poxa, continuo sendo cobrado por usuários que removi há meses. Estou cansado de pedir. Por favor, resolvam isso de uma vez.", "billing", 1, False),
]

gold = pd.DataFrame(LABELED_TICKETS, columns=["ticket", "gold_dept", "gold_frus", "gold_urg"])
gold["id"] = range(1, len(gold) + 1)
print(f"{len(gold)} tickets | distribuição de departamentos:")
print(gold.gold_dept.value_counts().to_string())

### 6.2 Abordagem A: Jev (com medição de tempo e tokens)

O Jev devolve o consumo de tokens em `response.usage`. Medimos o tempo de cada chamada com `time.perf_counter()`.

In [ ]:
def _usage_field(usage, name):
    if usage is None:
        return None
    return usage.get(name) if isinstance(usage, dict) else getattr(usage, name, None)


def run_jev(ticket: str) -> dict:
    t0 = time.perf_counter()
    r = client.system_one(state=ticket, questions=QUESTIONS)
    latency = time.perf_counter() - t0

    dept = r.choices["department"]
    usage = getattr(r, "usage", None)
    return {
        "dept": dept.choice,
        "dept_conf": dept.confidence,
        "frus": r.scores["frustration"].score,
        "urg": r.nouls["is_urgent"].noul,
        "lat": latency,
        "in": _usage_field(usage, "input_tokens"),
        "out": _usage_field(usage, "output_tokens"),
    }

### 6.3 Abordagem B: LLM com Structured Outputs

Para ser justo, o LLM recebe **a mesma rubrica** que o Jev (mesmas opções de departamento, mesma escala de frustração, mesma definição de urgência) e é forçado a responder no schema abaixo. Os tokens vêm de `completion.usage`.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class LLMTriage(BaseModel):
    department: Literal["billing", "technical", "sales"] = Field(
        description="Which team should handle this: billing = payment or subscription issues; "
                    "technical = bugs or integration problems; sales = pricing or account questions."
    )
    frustration: Literal[0, 1, 2] = Field(
        description="How frustrated the customer appears: 0 = calm, just stating facts; "
                    "1 = frustrated but civil; 2 = very angry, strong language."
    )
    urgency: float = Field(
        description="Probability, between 0.0 and 1.0, that the message conveys urgency or time-sensitivity."
    )

LLM_SYSTEM_PROMPT = "You analyze customer support tickets and fill in the requested fields."
CLASS_MODEL = "gpt-6-luna"

# Se trocar para um modelo de raciocínio (o1, gpt-5, ...), remova o parâmetro temperature.
def run_llm(ticket: str) -> dict:
    t0 = time.perf_counter()
    completion = openai_client.chat.completions.parse(
        model=CLASS_MODEL,
        messages=[
            {"role": "system", "content": LLM_SYSTEM_PROMPT},
            {"role": "user", "content": ticket},
        ],
        response_format=LLMTriage,
    )
    latency = time.perf_counter() - t0

    parsed = completion.choices[0].message.parsed
    if parsed is None:  # recusa ou falha de parsing
        raise RuntimeError(f"Sem saída estruturada: {completion.choices[0].message.refusal}")
    return {
        "dept": parsed.department,
        "frus": float(parsed.frustration),
        "urg": min(max(parsed.urgency, 0.0), 1.0),
        "lat": latency,
        "in": completion.usage.prompt_tokens,
        "out": completion.usage.completion_tokens,
    }

### 6.4 Executando as duas abordagens

Fazemos uma chamada de aquecimento (descartada) para cada uma, para que a primeira conexão não distorça a latência. As chamadas são **sequenciais**, o que mede a latência individual de cada uma.

In [ ]:
# Aquecimento (resultado descartado)
run_jev(gold.ticket.iloc[0])
run_llm(gold.ticket.iloc[0])

rows = []
for _, g in gold.iterrows():
    j = run_jev(g.ticket)
    l = run_llm(g.ticket)
    rows.append({
        "id": g.id,
        "gold_dept": g.gold_dept, "gold_frus": g.gold_frus, "gold_urg": g.gold_urg,
        "jev_dept": j["dept"], "jev_conf": j["dept_conf"], "jev_frus": j["frus"], "jev_urg": j["urg"],
        "jev_lat": j["lat"], "jev_in": j["in"], "jev_out": j["out"],
        "llm_dept": l["dept"], "llm_frus": l["frus"], "llm_urg": l["urg"],
        "llm_lat": l["lat"], "llm_in": l["in"], "llm_out": l["out"],
    })

df = pd.DataFrame(rows)
print("Concluído:", len(df), "tickets processados pelas duas abordagens.")

### 6.5 Comparação dos resultados

Primeiro, ticket a ticket (✓ = bate com o gabarito):

In [ ]:
def mark(pred, gold_value):
    return f"{pred} {'✓' if pred == gold_value else '✗'}"

view = pd.DataFrame({
    "id": df.id,
    "gabarito": df.gold_dept + " | F" + df.gold_frus.astype(str) + " | " + df.gold_urg.map({True: "urgente", False: "normal"}),
    "Jev dept": [mark(p, g) for p, g in zip(df.jev_dept, df.gold_dept)],
    "Jev conf.": df.jev_conf.round(2),
    "LLM dept": [mark(p, g) for p, g in zip(df.llm_dept, df.gold_dept)],
    "Jev frust.": df.jev_frus.round(2),
    "LLM frust.": df.llm_frus,
    "Jev urg.": df.jev_urg.round(2),
    "LLM urg.": df.llm_urg.round(2),
})
view

Agora as métricas agregadas. Como o Jev devolve valores contínuos (`score` de 0 a 2, `noul` de 0 a 1), além da acurácia usamos métricas que medem a **qualidade do valor numérico**:

- **Frustração:** erro médio absoluto contra o gabarito (menor = melhor).
- **Urgência:** acurácia com limiar de 0,5 e **Brier score** (erro quadrático médio da probabilidade; menor = melhor, e premia probabilidades bem calibradas).

In [ ]:
def summarize(prefix: str) -> dict:
    y_urg = df.gold_urg.astype(float)
    p_urg = df[f"{prefix}_urg"]
    lat = df[f"{prefix}_lat"]
    return {
        "QUALIDADE | Departamento: acurácia": (df[f"{prefix}_dept"] == df.gold_dept).mean(),
        "QUALIDADE | Frustração: erro médio absoluto (0–2)": (df[f"{prefix}_frus"] - df.gold_frus).abs().mean(),
        "QUALIDADE | Frustração: acerto da faixa (arredondado)": (df[f"{prefix}_frus"].round() == df.gold_frus).mean(),
        "QUALIDADE | Urgência: acurácia (limiar 0,5)": ((p_urg >= 0.5) == df.gold_urg).mean(),
        "QUALIDADE | Urgência: Brier score": ((p_urg - y_urg) ** 2).mean(),
        "TEMPO | Latência média (s)": lat.mean(),
        "TEMPO | Latência mediana (s)": lat.median(),
        "TEMPO | Latência p95 (s)": lat.quantile(0.95),
        "TEMPO | Tempo total (s)": lat.sum(),
    }

summary = pd.DataFrame({"Jev": summarize("jev"), "LLM (structured output)": summarize("llm")}).round(3)
summary

**Concordância entre as duas abordagens** e os casos em que divergem:

In [ ]:
agree_dept = (df.jev_dept == df.llm_dept).mean()
frus_gap = (df.jev_frus - df.llm_frus).abs().mean()
urg_gap = (df.jev_urg - df.llm_urg).abs().mean()

print(f"Mesmo departamento:                  {agree_dept:.0%}")
print(f"Diferença média de frustração (0–2): {frus_gap:.2f}")
print(f"Diferença média de urgência (0–1):   {urg_gap:.2f}")

divergent = df[df.jev_dept != df.llm_dept][["id", "gold_dept", "jev_dept", "jev_conf", "llm_dept"]]
print("\nTickets em que os departamentos divergem:")
display(divergent if len(divergent) else "nenhum")

💡 Repare na coluna `jev_conf`: o Jev informa o quanto confia na própria classificação, e o LLM com structured output **não tem equivalente nativo** (o JSON sempre vem "confiante"). Se os erros do Jev estiverem concentrados em confiança baixa, dá para usar o limiar de confiança como gatilho de fallback (como fizemos na seção 4).

### 6.6 Visualização

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
labels = ["Jev", "LLM"]
colors = ["#4C78A8", "#F58518"]

# Latência
ax = axes[0]
means = [df.jev_lat.mean(), df.llm_lat.mean()]
p95 = [df.jev_lat.quantile(0.95), df.llm_lat.quantile(0.95)]
x = np.arange(2)
ax.bar(x - 0.2, means, 0.4, label="média", color=colors)
ax.bar(x + 0.2, p95, 0.4, label="p95", color=colors, alpha=0.5)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_title("Latência por chamada (s)"); ax.legend()

# Tokens
ax = axes[1]
ins = [df.jev_in.mean(), df.llm_in.mean()]
outs = [df.jev_out.mean(), df.llm_out.mean()]
ax.bar(labels, ins, label="entrada", color=colors)
ax.bar(labels, outs, bottom=ins, label="saída", color=colors, alpha=0.5)
ax.set_title("Tokens por chamada (média)"); ax.legend()

# Qualidade
ax = axes[2]
metrics = {
    "Departamento\n(acurácia)": ("QUALIDADE | Departamento: acurácia"),
    "Urgência\n(acurácia)": ("QUALIDADE | Urgência: acurácia (limiar 0,5)"),
    "Frustração\n(faixa)": ("QUALIDADE | Frustração: acerto da faixa (arredondado)"),
}
x = np.arange(len(metrics))
ax.bar(x - 0.2, [summary.loc[k, "Jev"] for k in metrics.values()], 0.4, label="Jev", color=colors[0])
ax.bar(x + 0.2, [summary.loc[k, "LLM (structured output)"] for k in metrics.values()], 0.4, label="LLM", color=colors[1])
ax.set_xticks(x); ax.set_xticklabels(list(metrics.keys()))
ax.set_ylim(0, 1.05); ax.set_title("Qualidade vs. gabarito"); ax.legend()

plt.tight_layout()
plt.show()

### 6.7 Custo estimado (opcional)

Preencha os preços por 1 milhão de tokens com os valores atuais das páginas de preço de cada provedor. Deixei `None` onde eu não quis assumir um valor. Segundo a documentação de integração do Jev, a cobrança é feita **apenas sobre os tokens de entrada**.

In [ ]:
# Preços em USD por 1M de tokens. CONFIRA os valores atuais antes de usar.
PRICES = {
    "jev": {"input": 0.042, "output": 0.0},   # Jev cobra só entrada: preencha o preço de entrada
    "llm": {"input": 0.15, "output": 0.6},  # ex.: preços do modelo definido em OPENAI_MODEL
}

def estimate_cost(prefix: str):
    p = PRICES[prefix]
    if p["input"] is None or p["output"] is None:
        return None
    return (df[f"{prefix}_in"].sum() * p["input"] + df[f"{prefix}_out"].sum() * p["output"]) / 1_000_000

for prefix, name in [("jev", "Jev"), ("llm", "LLM")]:
    cost = estimate_cost(prefix)
    if cost is None:
        print(f"{name}: preencha PRICES['{prefix}'] para calcular o custo.")
    else:
        print(f"{name}: US$ {cost:.5f} no experimento | US$ {cost / len(df) * 1000:.3f} por 1.000 tickets")

### 6.8 Como interpretar (e o que cuidar)

- **Tokens não são 100% comparáveis.** Jev e OpenAI usam tokenizadores diferentes, e o LLM paga o custo do schema JSON e do prompt com a rubrica a cada chamada. Compare ordem de grandeza e, principalmente, **custo em dinheiro**.
- **Latência varia com rede, região e carga.** Os números vêm do Colab, chamadas sequenciais e uma única rodada. Para algo robusto, repita o experimento várias vezes, em horários diferentes, e compare também em paralelo.
- **Qualidade depende do domínio.** Com 16 exemplos, uma ou duas respostas mudam muito as porcentagens. Use um conjunto maior, rotulado pelo seu time.
- **Variantes do baseline LLM que valem testar:** um modelo maior, um campo de raciocínio (`reasoning`) antes das respostas (melhora a qualidade, mas aumenta tokens e latência), exemplos *few-shot* no prompt, e pedir probabilidades por `logprobs` em vez de um `float` gerado.
- **O que o formato dá de graça:** com o Jev você recebe `confidence` e `probabilities` calibradas por pergunta; com o LLM, a "probabilidade" de urgência é um número gerado pelo modelo, sem garantia de calibração.

## 7. Por que separar assim?

- **Previsível e testável:** a triagem é um objeto tipado. Dá para escrever testes, logar, medir distribuições e criar alertas (ex.: `urgency > 0.9`) sem parsear texto.
- **Controle explícito:** as regras de negócio (limiares de tom, urgência, fallback por baixa confiança) ficam no seu código, não escondidas num prompt gigante.
- **Calibração:** o Jev devolve probabilidades e `confidence`, o que permite decidir *quando não confiar* na classificação (veja a regra de `department_confidence` acima).
- **Responsabilidades claras:** o modelo System One decide/classifica; o LLM generativo só escreve.